# Quantificare l'Eredità Progettuale Condivisa in una Flotta di Trasformatori di Potenza con PROC INBREED


## Riepilogo Esecutivo

I trasformatori di sottostazione di un gestore di rete sono progettati in generazioni successive, ogni nuovo modello derivato da due progetti predecessori. Questo notebook tratta questa genealogia ingegneristica come un pedigree e usa **PROC INBREED** per calcolare i coefficienti di consanguineità e di parentela additiva (covarianza), quantificando quanta eredità progettuale condividono due asset qualsiasi.

Il pedigree è costruito in modo che la struttura sia interpretabile: due dei quattro modelli dell'attuale flotta discendono da una coppia di progetti predecessori **fratelli** e quindi portano un'eredità concentrata, mentre gli altri attingono a lignaggi disgiunti. La procedura recupera esattamente questo risultato. I due modelli derivati da fratelli (`G3_T1`, `G3_T3`) portano ciascuno un coefficiente di consanguineità di **F = 0.25**; gli altri due (`G3_T2`, `G3_T4`) hanno **F = 0**. Il coefficiente di consanguineità medio della flotta è **0.0417**. Leggendo la matrice di parentela additiva, la coppia meno correlata dell'attuale flotta è **`G3_T1` e `G3_T3` (parentela = 0)** — non condividono alcuna ascendenza e costituiscono l'abbinamento ridondante (N-1) più solido, il che è rilevante perché `G3_T1` è essa stessa uno dei progetti più consanguinei.


## Origini Dati

| Dataset | Righe | Variabili Chiave | Descrizione |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Un pedigree ingegneristico piccolo e deterministico di progetti di trasformatori di sottostazione su tre generazioni di progettazione non sovrapposte (4 piattaforme fondatrici, 4 derivati di seconda generazione, 4 modelli dell'attuale flotta). Ogni progetto non fondatore registra i due progetti predecessori distinti (`Parent1`, `Parent2`) da cui è stato derivato. `Sex` etichetta il ruolo di progettazione guida (M = lignaggio meccanico/nucleo, F = lignaggio elettrico/avvolgimento). Il pedigree è specificato manualmente — non casuale — cosicché la struttura di parentela è nota in anticipo e può essere verificata rispetto all'output della procedura. |


# Quantificare l'Eredità Progettuale Condivisa in una Flotta di Trasformatori di Potenza

## Perché un'utility si preoccupa della "consanguineità"

Un operatore di trasmissione e distribuzione gestisce centinaia di trasformatori di potenza di sottostazione. Per controllare i costi ingegneristici e lo sforzo di qualifica, i produttori raramente progettano ogni trasformatore da zero — ogni nuovo modello **eredita** geometria del nucleo, topologia degli avvolgimenti, sistemi di isolamento e progetti dei passanti da uno o due modelli predecessori. Nel corso di diversi cicli di approvvigionamento questo produce una vera e propria *genealogia ingegneristica*: un pedigree di progetti.

Questa eredità condivisa è un rischio nascosto per l'affidabilità. Se due trasformatori che proteggono lo stesso carico (una coppia ridondante **N-1**) discendono dallo stesso progetto ancestrale, un difetto di progettazione latente — un modo di risonanza, un meccanismo di invecchiamento dell'isolamento, un percorso di scarica dei passanti — è probabile che sia presente in **entrambe** le unità. Un'unica causa radice può quindi mettere fuori servizio simultaneamente la coppia ridondante: un *guasto a modo comune*.

**PROC INBREED** è stata costruita per quantificare esattamente questo tipo di ascendenza condivisa. Sebbene le sue origini siano nell'allevamento animale e vegetale, la sua matematica — la ricorsione di parentela additiva di Wright — è agnostica rispetto al dominio: misura la frazione di eredità progettuale che due individui condividono attraverso antenati comuni. Mappiamo il vocabolario genetico sull'ingegneria degli asset:

| Concetto INBREED | Interpretazione per l'utility |
|---|---|
| Individuo | Un progetto/modello di trasformatore |
| Genitore (padre/madre) | Un progetto predecessore da cui è stato derivato |
| Generazione (CLASS) | Un ciclo di approvvigionamento/progettazione |
| Coefficiente di consanguineità *F* | Grado di eredità auto-simile all'interno di un progetto |
| Coefficiente di covarianza/parentela | Eredità progettuale condivisa tra due progetti |
| Coppia meno correlata | Il miglior abbinamento ridondante per la resilienza N-1 |


## Passo 1 — Specificare il pedigree di progettazione

Inseriamo `DesignLineage` direttamente in modo che la struttura di parentela sia inequivocabile. Ci sono tre **generazioni di progettazione non sovrapposte**:

- **Generazione 1** — quattro progetti di piattaforma fondatrice (`G1_P1`-`G1_P4`) senza predecessori (genitori vuoti).
- **Generazione 2** — quattro progetti derivati (`G2_D1`-`G2_D4`), ciascuno ingegnerizzato da due piattaforme di Generazione 1 **distinte**. `G2_D1` e `G2_D2` sono *fratelli germani* (entrambi da `G1_P1` x `G1_P2`); `G2_D3` e `G2_D4` sono fratelli germani (entrambi da `G1_P3` x `G1_P4`).
- **Generazione 3** — quattro modelli dell'attuale flotta (`G3_T1`-`G3_T4`). `G3_T1` è costruito dalla coppia di fratelli `G2_D1` x `G2_D2`, e `G3_T3` dalla coppia di fratelli `G2_D3` x `G2_D4`; questi due progetti concentrano quindi l'eredità. `G3_T2` e `G3_T4` incrociano ciascuno i due lignaggi disgiunti.

Le colonne `ID`, `Parent1` e `Parent2` portano il pedigree; `Sex` registra quale lignaggio ingegneristico ha guidato il progetto. Un breve DATA step successivo azzera il segnaposto `.` in modo che le piattaforme fondatrici abbiano genitori vuoti, come richiesto da PROC INBREED.


In [1]:
DATI DesignLineage;
   LUNGHEZZA ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   INGRESSO Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
ESEGUIRE;

/* Le piattaforme fondatrici non hanno predecessori: azzera i punti segnaposto */
DATI DesignLineage;
   IMPOSTARE DesignLineage;
   SE_COND Parent1 = '.' ALLORA Parent1 = ' ';
   SE_COND Parent2 = '.' ALLORA Parent2 = ' ';
ESEGUIRE;

TITOLO 'Pedigree di Progettazione dei Trasformatori';
PROCEDURA STAMPARE DATI=DesignLineage noobs;
   VARIABILE Generation ID Parent1 Parent2 Sex;
   ETICHETTA Generation='Generazione' ID='Identificativo'
       Parent1='Predecessore 1' Parent2='Predecessore 2' Sex='Ruolo';
ESEGUIRE;


                                      Pedigree di Progettazione dei Trasformatori                                       


Generazione  Identificativo  Predecessore 1  Predecessore 2  Ruolo
-----------  --------------  --------------  --------------  -----
          1  G1_P1                                           M
          1  G1_P2                                           M
          1  G1_P3                                           M
          1  G1_P4                                           F
          2  G2_D1           G1_P1           G1_P2           M
          2  G2_D2           G1_P1           G1_P2           F
          2  G2_D3           G1_P3           G1_P4           F
          2  G2_D4           G1_P3           G1_P4           M
          3  G3_T1           G2_D1           G2_D2           M
          3  G3_T2           G2_D1           G2_D3           F
          3  G3_T3           G2_D3           G2_D4           F
          3  G3_T4           G2_D2           G2_D4


NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Pedigree di Progettazione dei Trasformatori.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Passo 2 — Coefficienti di consanguineità per generazione di progettazione

Eseguiamo PROC INBREED in **modalità multi-generazione** indicando `Generation` nell'istruzione `CLASS`, che stampa il riepilogo del pedigree per generazione. L'istruzione `VAR` fornisce le tre colonne del pedigree nell'ordine richiesto: individuo, primo predecessore, secondo predecessore.

- L'opzione **IND** stampa il coefficiente di consanguineità *F* di ogni progetto — una misura di quanto sia concentrata la sua eredità. Un progetto costruito da due predecessori strettamente imparentati porta un *F* elevato.
- L'opzione **MATRIX** stampa la matrice completa di parentela additiva così da poter leggere direttamente l'eredità a coppie.
- L'opzione **AVERAGE** aggiunge il coefficiente di consanguineità medio dell'intera flotta — un unico riepilogo della diversità progettuale.

Valori vicini a 0 indicano lignaggi indipendenti; *F* aumenta quando i due predecessori di un progetto diventano più strettamente imparentati.


In [2]:
TITOLO 'Coefficienti di Consanguineità per Generazione di Progettazione';
PROCEDURA inbreed DATI=DesignLineage ind average MATRIX;
   VARIABILE ID Parent1 Parent2;
   CLASSE Generation;
ESEGUIRE;


                            Coefficienti di Consanguineità per Generazione di Progettazione                             


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Coefficienti di Consanguineità per Generazione di Progettazione.
NOTE: PROC INBREED data=DesignLineage



## Passo 3 — Coefficienti di covarianza salvati per la valutazione del rischio a valle

Sostituendo la vista di consanguineità con l'opzione **COVAR** si ottengono le stesse relazioni espresse come **coefficienti di covarianza (parentela additiva)**, la forma che un team di asset management userebbe per alimentare un punteggio di rischio di ridondanza. L'opzione **OUTCOV=** cattura la matrice completa come dataset (`DesignCovar`), dove le colonne `Col1`-`Col12` contengono la parentela di ogni progetto rispetto a ogni altro progetto (nell'ordine del pedigree) — pronta per essere ricongiunta alle assegnazioni di sottostazione.

Stampiamo il dataset di output in modo che la matrice salvata sia visibile accanto al listato.


In [3]:
TITOLO 'Coefficienti di Covarianza (Parentela)';
PROCEDURA inbreed DATI=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VARIABILE ID Parent1 Parent2;
   CLASSE Generation;
ESEGUIRE;

TITOLO 'Matrice di Parentela OUTCOV= Salvata per la Valutazione del Rischio';
PROCEDURA STAMPARE DATI=DesignCovar noobs;
ESEGUIRE;


                                         Coefficienti di Covarianza (Parentela)                                         


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Coefficienti di Covarianza (Parentela).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Matrice di Parentela OUTCOV= Salvata per la Valutazione del Rischio.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Passo 4 — Abbinamenti meno correlati per installazioni ridondanti (N-1)

La matrice `DesignCovar` salvata è esattamente ciò di cui necessita uno studio di ridondanza. La parentela del progetto *i* rispetto al progetto *j* si trova nella riga *i*, colonna `Col`*j* (le colonne sono nell'ordine del pedigree, quindi `Col9`-`Col12` corrispondono ai quattro modelli dell'attuale flotta `G3_T1`-`G3_T4`).

Leggiamo le quattro righe dell'attuale flotta da `DesignCovar`, formiamo ogni coppia non ordinata dell'attuale flotta, associamo il relativo coefficiente di parentela e ordiniamo dal meno correlato. L'obiettivo è scegliere le coppie ridondanti la cui eredità condivisa è **minima** — questo riduce al minimo la probabilità che un difetto di progettazione ereditato disabiliti entrambe le metà di una posizione protetta N-1.


In [4]:
/* Estrai le quattro righe dell'attuale flotta; Col9..Col12 sono le
   parentele verso G3_T1..G3_T4 (ordine colonne del pedigree).
   Costruisci ogni coppia non ordinata. */
DATI Pairs;
   IMPOSTARE DesignCovar;
   DOVE ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   LUNGHEZZA DesignA $12 DesignB $12;
   VETTORE g3{4} Col9 Col10 Col11 Col12;
   VETTORE nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   FARE j = 1 FINO_A 4;
      DesignB = nm{j};
      SE_COND DesignA < DesignB ALLORA FARE;
         Relationship = g3{j};
         USCITA;
      FINE;
   FINE;
   MANTENERE DesignA DesignB Relationship;
ESEGUIRE;

PROCEDURA ORDINARE DATI=Pairs; PER Relationship; ESEGUIRE;

TITOLO 'Abbinamenti Ridondanti Candidati (N-1), dal Meno Correlato';
PROCEDURA STAMPARE DATI=Pairs noobs;
   VARIABILE DesignA DesignB Relationship;
   ETICHETTA DesignA='Progetto A' DesignB='Progetto B'
       Relationship='Grado di Parentela';
ESEGUIRE;
TITOLO;


                               Abbinamenti Ridondanti Candidati (N-1), dal Meno Correlato                               


Progetto A  Progetto B  Grado di Parentela
----------  ----------  ------------------
G3_T1       G3_T3                        0
G3_T2       G3_T4                     0.25
G3_T1       G3_T2                    0.375
G3_T1       G3_T4                    0.375
G3_T2       G3_T3                    0.375
G3_T3       G3_T4                    0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Abbinamenti Ridondanti Candidati (N-1), dal Meno Correlato.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretazione dei risultati

**Coefficienti di consanguineità (Passo 2).** Tutte le piattaforme fondatrici di Generazione 1 e tutti i derivati di Generazione 2 mostrano *F* = 0 — per costruzione nessuno ha due predecessori imparentati. Due modelli di Generazione 3 emergono con *F* = 0.25: `G3_T1` (costruito dalla coppia di fratelli `G2_D1` x `G2_D2`) e `G3_T3` (dalla coppia di fratelli `G2_D3` x `G2_D4`). I loro predecessori risalgono alla *stessa* coppia di piattaforme fondatrici, quindi la loro eredità è concentrata. Dal punto di vista dell'affidabilità questi sono i progetti più esposti a un singolo difetto ereditato, e richiedono test di qualifica indipendenti aggiuntivi. `G3_T2` e `G3_T4`, che incrociano i due lignaggi disgiunti, hanno *F* = 0.

**Matrice di parentela (Passo 3).** Le voci fuori diagonale quantificano l'eredità condivisa a coppie. Le due coppie di fratelli di Generazione 2 mostrano ciascuna una parentela di 0.50 reciproca (`G2_D1`-`G2_D2` e `G2_D3`-`G2_D4`), mentre i derivati di lignaggi opposti si collocano a 0.00. I progetti consanguinei dell'attuale flotta portano auto-parentele di 1.25 (= 1 + *F*), visibili sulla diagonale. Il dataset `DesignCovar` rende disponibile la matrice completa per la ricongiunzione con le assegnazioni di sottostazione.

**Abbinamenti meno correlati (Passo 4).** Classificando ogni coppia dell'attuale flotta per parentela si ottiene **`G3_T1` e `G3_T3` al primo posto con parentela 0.00** — discendono da piattaforme ancestrali interamente disgiunte e non condividono alcuna eredità progettuale. Questo è l'abbinamento ridondante più solido, ed è particolarmente prezioso perché `G3_T1` è essa stessa uno dei due progetti più consanguinei: abbinarla a una controparte completamente non imparentata compensa la sua eredità concentrata. La coppia successiva per merito è `G3_T2` e `G3_T4` a 0.25; le rimanenti coppie si collocano a 0.375. Il coefficiente di consanguineità medio della flotta di **0.0417** (stampato dall'opzione AVERAGE nel Passo 2) riassume la diversità progettuale complessiva. L'approvvigionamento dovrebbe preferire le coppie a minore parentela per le posizioni N-1 e, nel tempo, ampliare la base ancestrale — l'equivalente, nell'ingegneria degli asset, dell'evitare un collo di bottiglia genetico.

**Avvertenza.** Questi sono dati sintetici illustrativi; uno studio di produzione costruirebbe il pedigree a partire dai registri reali di revisione di progettazione del produttore e convaliderebbe i punteggi di parentela rispetto a eventi storici di guasto a modo comune prima di guidare le decisioni di approvvigionamento.
